In [ ]:
# ===================== 0. IMPORT THƯ VIỆN =====================
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms, datasets
from efficientnet_pytorch import EfficientNet

from PIL import Image
import numpy as np

# ===================== 1. CẤU HÌNH CHUNG =====================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Đường dẫn dữ liệu (để lấy class_names cho đúng 22 lớp)
train_data_dir = r"E:\archive\SkinDisease\SkinDisease\train"

VGG_CKPT_PATH      = r"E:\archive\PiplineCode pretrain=True\vgg16_pretrained_fulloption.pth"
RES50_CKPT_PATH    = r"E:\archive\PiplineCode pretrain=True\resnet50_pretrained_fulloption.pth"
EFFB4_CKPT_PATH    = r"E:\archive\PiplineCode pretrain=True\efficientnet_b4_pretrained_fulloption.pth"
EFFB0_CKPT_PATH    = r"E:\archive\PiplineCode pretrain=True\efficientnet_b0_pretrained_fulloption.pth"
DENSE121_CKPT_PATH = r"E:\archive\PiplineCode pretrain=True\densenet121_pretrained_fulloption.pth"
MOBILEV3_CKPT_PATH = r"E:\archive\PiplineCode pretrain=True\mobilenetv3_fulloption_imagenet.pth"

IMAGE_SIZE    = 300
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ===================== 2. LẤY DANH SÁCH CLASS TỪ THƯ MỤC TRAIN =====================
tmp_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor()
])
train_ds = datasets.ImageFolder(train_data_dir, transform=tmp_transform)

class_names = train_ds.classes
num_classes = len(class_names)

print(f"📂 Số lớp (lấy từ folder train): {num_classes}")
print("Danh sách lớp:", class_names)

# Transform dùng cho inference
inference_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# ===================== 3. HÀM KHỞI TẠO & LOAD CÁC MODEL =====================
def load_vgg16(num_classes, ckpt_path):
    try:
        weights = models.VGG16_Weights.IMAGENET1K_V1
        model = models.vgg16(weights=weights)
    except AttributeError:  # phiên bản torchvision cũ
        model = models.vgg16(pretrained=True)

    in_features = model.classifier[6].in_features
    model.classifier[6] = nn.Linear(in_features, num_classes)

    state = torch.load(ckpt_path, map_location="cpu")
    model.load_state_dict(state)
    model.to(device)
    model.eval()
    print("✅ VGG16 loaded.")
    return model

def load_resnet50(num_classes, ckpt_path):
    try:
        weights = models.ResNet50_Weights.IMAGENET1K_V1
        model = models.resnet50(weights=weights)
    except AttributeError:
        model = models.resnet50(pretrained=True)

    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)

    state = torch.load(ckpt_path, map_location="cpu")
    model.load_state_dict(state)
    model.to(device)
    model.eval()
    print("✅ ResNet50 loaded.")
    return model

def load_efficientnet_b4(num_classes, ckpt_path):
    model = EfficientNet.from_pretrained('efficientnet-b4', num_classes=num_classes)
    state = torch.load(ckpt_path, map_location="cpu")
    model.load_state_dict(state)
    model.to(device)
    model.eval()
    print("✅ EfficientNet-B4 loaded.")
    return model

def load_efficientnet_b0(num_classes, ckpt_path):
    model = EfficientNet.from_pretrained('efficientnet-b0', num_classes=num_classes)
    state = torch.load(ckpt_path, map_location="cpu")
    model.load_state_dict(state)
    model.to(device)
    model.eval()
    print("✅ EfficientNet-B0 loaded.")
    return model

def load_densenet121(num_classes, ckpt_path):
    try:
        weights = models.DenseNet121_Weights.IMAGENET1K_V1
        model = models.densenet121(weights=weights)
    except AttributeError:
        model = models.densenet121(pretrained=True)

    in_features = model.classifier.in_features
    model.classifier = nn.Linear(in_features, num_classes)

    state = torch.load(ckpt_path, map_location="cpu")
    model.load_state_dict(state)
    model.to(device)
    model.eval()
    print("✅ DenseNet121 loaded.")
    return model

def load_mobilenet_v3(num_classes, ckpt_path):
    try:
        weights = models.MobileNet_V3_Large_Weights.IMAGENET1K_V1
        model = models.mobilenet_v3_large(weights=weights)
    except AttributeError:
        model = models.mobilenet_v3_large(pretrained=True)

    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, num_classes)

    state = torch.load(ckpt_path, map_location="cpu")
    model.load_state_dict(state)
    model.to(device)
    model.eval()
    print("✅ MobileNetV3-Large loaded.")
    return model

# ===================== 4. KHỞI TẠO 6 MÔ HÌNH (22 LỚP) =====================
vgg16_model      = load_vgg16(num_classes, VGG_CKPT_PATH)
resnet50_model   = load_resnet50(num_classes, RES50_CKPT_PATH)
effb4_model      = load_efficientnet_b4(num_classes, EFFB4_CKPT_PATH)
effb0_model      = load_efficientnet_b0(num_classes, EFFB0_CKPT_PATH)
densenet_model   = load_densenet121(num_classes, DENSE121_CKPT_PATH)
mobilenet_model  = load_mobilenet_v3(num_classes, MOBILEV3_CKPT_PATH)

# ===================== 5. HÀM DỰ ĐOÁN CHO 1 MÔ HÌNH =====================
def predict_single_model(model, img_tensor):
    """
    img_tensor: [1, 3, H, W] (đã transform + to(device))
    return: (pred_idx, pred_prob, prob_vec_numpy)
    """
    with torch.no_grad():
        outputs = model(img_tensor)           # [1, num_classes]
        probs   = F.softmax(outputs, dim=1)   # [1, num_classes]
        probs_np = probs.cpu().numpy()[0]
        pred_idx = int(np.argmax(probs_np))
        pred_prob = float(probs_np[pred_idx])
    return pred_idx, pred_prob, probs_np

# ===================== 6. SOFT VOTING 6 MODEL =====================
def soft_voting_predict(
    image_path,
    vgg_model,
    res_model,
    effb4_model,
    effb0_model,
    dense_model,
    mobile_model,
    weights=(1.0, 1.0, 1.0, 1.0, 1.0, 1.0)
):
    """
    Soft voting 6 model dựa trên trung bình vector xác suất.
    weights: (w_vgg, w_resnet, w_effb4, w_effb0, w_dense, w_mobile)
    """
    # --------- Đọc & tiền xử lý ảnh ---------
    img_pil = Image.open(image_path).convert("RGB")
    x = inference_transform(img_pil).unsqueeze(0).to(device)   # [1, 3, H, W]

    # --------- Dự đoán từng mô hình ---------
    vgg_idx, vgg_prob, vgg_vec     = predict_single_model(vgg_model, x)
    res_idx, res_prob, res_vec     = predict_single_model(res_model, x)
    effb4_idx, effb4_prob, effb4_vec = predict_single_model(effb4_model, x)
    effb0_idx, effb0_prob, effb0_vec = predict_single_model(effb0_model, x)
    dense_idx, dense_prob, dense_vec = predict_single_model(dense_model, x)
    mob_idx, mob_prob, mob_vec       = predict_single_model(mobile_model, x)

    # --------- Soft Voting (trung bình có trọng số) ---------
    w_vgg, w_res, w_effb4, w_effb0, w_dense, w_mob = weights
    total_w = w_vgg + w_res + w_effb4 + w_effb0 + w_dense + w_mob

    avg_probs = (
        w_vgg   * vgg_vec   +
        w_res   * res_vec   +
        w_effb4 * effb4_vec +
        w_effb0 * effb0_vec +
        w_dense * dense_vec +
        w_mob   * mob_vec
    ) / total_w

    final_idx  = int(np.argmax(avg_probs))
    final_prob = float(avg_probs[final_idx])

    # ===================== 7. IN KẾT QUẢ =====================
    print(f"📌 Ảnh đầu vào: {image_path}\n")

    print("🔹 Kết quả từng mô hình:")
    print(f"  - VGG16          → {class_names[vgg_idx]} ({vgg_prob*100:.2f}%)")
    print(f"  - ResNet50       → {class_names[res_idx]} ({res_prob*100:.2f}%)")
    print(f"  - EfficientNet-B4→ {class_names[effb4_idx]} ({effb4_prob*100:.2f}%)")
    print(f"  - EfficientNet-B0→ {class_names[effb0_idx]} ({effb0_prob*100:.2f}%)")
    print(f"  - DenseNet121    → {class_names[dense_idx]} ({dense_prob*100:.2f}%)")
    print(f"  - MobileNetV3    → {class_names[mob_idx]} ({mob_prob*100:.2f}%)\n")

    print("🟩 Kết quả Soft Voting (cuối cùng):")
    print(f"  → {class_names[final_idx]}  ({final_prob*100:.2f}%)")

    # Trả về dict nếu muốn dùng tiếp
    return {
        "vgg":       {"idx": vgg_idx, "label": class_names[vgg_idx], "prob": vgg_prob},
        "resnet50":  {"idx": res_idx, "label": class_names[res_idx], "prob": res_prob},
        "effb4":     {"idx": effb4_idx, "label": class_names[effb4_idx], "prob": effb4_prob},
        "effb0":     {"idx": effb0_idx, "label": class_names[effb0_idx], "prob": effb0_prob},
        "densenet":  {"idx": dense_idx, "label": class_names[dense_idx], "prob": dense_prob},
        "mobilenet": {"idx": mob_idx, "label": class_names[mob_idx], "prob": mob_prob},
        "soft_vote": {"idx": final_idx, "label": class_names[final_idx], "prob": final_prob}
    }

# ===================== GỌI HÀM =====================
test_image_path = r"E:\archive\SkinDisease\AnhTest\Actinic_Keratosis\actinic-keratosis-5FU-56.jpeg"

results = soft_voting_predict(
    test_image_path,
    vgg_model=vgg16_model,
    res_model=resnet50_model,
    effb4_model=effb4_model,
    effb0_model=effb0_model,
    dense_model=densenet_model,
    mobile_model=mobilenet_model,
    weights=(1.0, 1.0, 1.0, 1.0, 1.0, 1.0)   # có thể chỉnh trọng số nếu muốn
)


Using device: cuda
📂 Số lớp (lấy từ folder train): 22
Danh sách lớp: ['Acne', 'Actinic_Keratosis', 'Benign_tumors', 'Bullous', 'Candidiasis', 'DrugEruption', 'Eczema', 'Infestations_Bites', 'Lichen', 'Lupus', 'Moles', 'Psoriasis', 'Rosacea', 'Seborrh_Keratoses', 'SkinCancer', 'Sun_Sunlight_Damage', 'Tinea', 'Unknown_Normal', 'Vascular_Tumors', 'Vasculitis', 'Vitiligo', 'Warts']


C:\Users\letra\AppData\Local\Temp\ipykernel_10208\1690553314.py:62: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(ckpt_path, map_location="cpu")


✅ VGG16 loaded.


C:\Users\letra\AppData\Local\Temp\ipykernel_10208\1690553314.py:79: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(ckpt_path, map_location="cpu")


✅ ResNet50 loaded.
Loaded pretrained weights for efficientnet-b4


...

✅ EfficientNet-B4 loaded.
Loaded pretrained weights for efficientnet-b0
✅ EfficientNet-B0 loaded.


C:\Users\letra\AppData\Local\Temp\ipykernel_10208\1690553314.py:97: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(ckpt_path, map_location="cpu")
C:\Users\

✅ DenseNet121 loaded.


C:\Users\letra\AppData\Local\Temp\ipykernel_10208\1690553314.py:131: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(ckpt_path, map_location="cpu")


✅ MobileNetV3-Large loaded.
📌 Ảnh đầu vào: E:\archive\SkinDisease\AnhTest\Actinic_Keratosis\actinic-keratosis-5FU-56.jpeg

🔹 Kết quả từng mô hình:
  - VGG16          → Actinic_Keratosis (99.61%)
  - ResNet50       → Rosacea (76.62%)
  - EfficientNet-B4→ Actinic_Keratosis (84.86%)
  - EfficientNet-B0→ Rosacea (49.58%)
  - DenseNet121    → Actinic_Keratosis (87.35%)
  - MobileNetV3    → Actinic_Keratosis (76.25%)

🟩 Kết quả Soft Voting (cuối cùng):
  → Actinic_Keratosis  (64.14%)


In [2]:
import os

# ===================== 8. SOFT VOTING CHO CẢ FOLDER ẢNH (6 MODEL) =====================
def soft_voting_predict_folder(folder_path,
                               vgg_model,
                               res_model,
                               effb4_model,
                               effb0_model,
                               dense_model,
                               mobile_model,
                               weights=(1.0, 1.0, 1.0, 1.0, 1.0, 1.0)):
    """
    Đọc toàn bộ ảnh trong 1 folder và in kết quả soft voting (6 model) cho TỪNG ẢNH.
    weights: (w_vgg, w_resnet, w_effb4, w_effb0, w_densenet, w_mobilenet)
    """

    exts = (".jpg", ".jpeg", ".png", ".bmp", ".gif")  # các đuôi ảnh cơ bản
    files = [f for f in os.listdir(folder_path)
             if f.lower().endswith(exts)]

    files.sort()  # cho gọn gàng

    if not files:
        print(f"⚠️ Folder không có ảnh: {folder_path}")
        return

    print(f"📁 Đang dự đoán cho folder: {folder_path}")
    print(f"➡️ Số ảnh tìm thấy: {len(files)}\n")

    # Nếu muốn tính độ chính xác theo label của tên folder:
    true_label_name = os.path.basename(folder_path.rstrip("\\/"))
    total = 0
    correct = 0

    for fname in files:
        img_path = os.path.join(folder_path, fname)
        print("===========================================")
        print(f"🖼️ File: {fname}")

        results = soft_voting_predict(
            img_path,
            vgg_model=vgg16_model,
            res_model=resnet50_model,
            effb4_model=effb4_model,
            effb0_model=effb0_model,
            dense_model=densenet_model,
            mobile_model=mobilenet_model,
            weights=weights
        )

        # So sánh với "label thật" lấy từ tên folder (nếu trùng class_names)
        pred_label = results["soft_vote"]["label"]
        if pred_label == true_label_name:
            correct += 1
        total += 1

        print()  # dòng trống cho dễ đọc

    # In thống kê cuối cùng (nếu folder là 1 lớp duy nhất)
    if total > 0:
        acc = correct / total * 100
        print("===========================================")
        print(f"✅ Tổng kết folder: {folder_path}")
        print(f"   Label (theo tên folder): {true_label_name}")
        print(f"   Đúng: {correct}/{total}  → Accuracy: {acc:.2f}%")


In [18]:
# ===================== 9. VÍ DỤ GỌI HÀM CHO CẢ FOLDER =====================
test_folder_path = r"E:\archive\SkinDisease\AnhTest\Seborrh_Keratoses"

soft_voting_predict_folder(
    test_folder_path,
    vgg_model=vgg16_model,
    res_model=resnet50_model,
    effb4_model=effb4_model,
    effb0_model=effb0_model,
    dense_model=densenet_model,
    mobile_model=mobilenet_model,
    weights=(1.0, 1.0, 1.0, 1.0, 1.0, 1.0)   
)


📁 Đang dự đoán cho folder: E:\archive\SkinDisease\AnhTest\Seborrh_Keratoses
➡️ Số ảnh tìm thấy: 7

🖼️ File: sebks08__ProtectWyJQcm90ZWN0Il0_FocusFillWzI5NCwyMjIsInkiLDVd.jpeg
📌 Ảnh đầu vào: E:\archive\SkinDisease\AnhTest\Seborrh_Keratoses\sebks08__ProtectWyJQcm90ZWN0Il0_FocusFillWzI5NCwyMjIsInkiLDVd.jpeg

🔹 Kết quả từng mô hình:
  - VGG16          → Seborrh_Keratoses (100.00%)
  - ResNet50       → Seborrh_Keratoses (100.00%)
  - EfficientNet-B4→ Seborrh_Keratoses (100.00%)
  - EfficientNet-B0→ Seborrh_Keratoses (99.99%)
  - DenseNet121    → Seborrh_Keratoses (97.99%)
  - MobileNetV3    → Actinic_Keratosis (93.22%)

🟩 Kết quả Soft Voting (cuối cùng):
  → Seborrh_Keratoses  (83.49%)

🖼️ File: sebks38__ProtectWyJQcm90ZWN0Il0_FocusFillWzI5NCwyMjIsInkiLDM2XQ.jpeg
📌 Ảnh đầu vào: E:\archive\SkinDisease\AnhTest\Seborrh_Keratoses\sebks38__ProtectWyJQcm90ZWN0Il0_FocusFillWzI5NCwyMjIsInkiLDM2XQ.jpeg

🔹 Kết quả từng mô hình:
  - VGG16          → Seborrh_Keratoses (66.77%)
  - ResNet50       → Sebo